In [ ]:
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


In [ ]:
%pip install -U pip
%pip install pandas numpy scikit-learn nibabel tqdm matplotlib torch torchvision torchaudio

In [ ]:
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix

In [ ]:
# =========================
# Paths
# =========================

CSV_PATH = REPO_ROOT / "data/csv/matched_cn_ad_labels_all.csv"
NIFTI_ROOT = REPO_ROOT / "data/nifti"

OUT_CSV = REPO_ROOT / "results/autorater/autorater_predictions_all_subject_oof.csv"

# =========================
# Training settings
# =========================

BATCH_SIZE = 2
EPOCHS = 5
LR = 1e-4
N_FOLDS = 5
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # More deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
def choose_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


# -------------------------
# 1) Human label H
# -------------------------

if "H" in df.columns:
    df["H"] = pd.to_numeric(df["H"], errors="coerce").astype(int)
else:
    label_col = choose_column(df, ["label", "Label", "Group", "Diagnosis", "DX"])
    if label_col is None:
        raise ValueError("Could not find label column. Need one of: label, Label, Group, Diagnosis, DX, or H.")
    
    print("Using label column:", label_col)
    df["H"] = (df[label_col].astype(str).str.upper().str.strip() == "AD").astype(int)


print("H counts:")
print(df["H"].value_counts(dropna=False))


# -------------------------
# 2) subject_id
# -------------------------

subject_col = choose_column(
    df,
    ["subject_id", "Subject", "subject", "SubjectID", "PTID", "RID", "participant_id", "PatientID"]
)

if subject_col is None:
    raise ValueError(
        "Could not find subject column. Please add a subject_id column or use one of: "
        "Subject, PTID, RID, participant_id."
    )

print("Using subject column:", subject_col)
df["subject_id"] = df[subject_col].astype(str).str.strip()


# -------------------------
# 3) nifti_path
# -------------------------

if "nifti_path" not in df.columns:
    raise ValueError("CSV must contain a nifti_path column.")

df["nifti_path"] = df["nifti_path"].astype(str).str.strip()

print("\nBasic summary")
print("Rows:", len(df))
print("Unique subjects:", df["subject_id"].nunique())
print("AD prevalence:", df["H"].mean())

In [ ]:
# Build NIfTI index once.
# This may take a little while depending on folder size.

print("Indexing NIfTI files under:", NIFTI_ROOT)
nifti_files = list(NIFTI_ROOT.rglob("*.nii")) + list(NIFTI_ROOT.rglob("*.nii.gz"))

print("Number of NIfTI files found:", len(nifti_files))
if len(nifti_files) == 0:
    raise FileNotFoundError(f"No .nii or .nii.gz files found under {NIFTI_ROOT}")

# filename -> path
filename_to_path = {}
for p in nifti_files:
    filename_to_path.setdefault(p.name, p)

print("Example NIfTI files:")
for p in nifti_files[:5]:
    print(p)

In [ ]:
def resolve_nifti_path(path_str, subject_id=None):
    """
    Resolve old or broken nifti_path to the current NIFTI_ROOT.

    Priority:
    1. Existing path as-is
    2. Same filename under NIFTI_ROOT
    3. Any NIfTI file containing subject_id in its path/name
    """
    p = Path(path_str)

    # 1) existing path
    if p.exists():
        return str(p)

    # 2) same filename in current NIFTI_ROOT
    if p.name in filename_to_path:
        return str(filename_to_path[p.name])

    # 3) search by subject_id
    if subject_id is not None:
        sid = str(subject_id).strip()
        matches = [q for q in nifti_files if sid in str(q)]
        if len(matches) > 0:
            return str(matches[0])

    return None


resolved_paths = []
missing_rows = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Resolving NIfTI paths"):
    resolved = resolve_nifti_path(row["nifti_path"], row["subject_id"])
    resolved_paths.append(resolved)

    if resolved is None:
        missing_rows.append(i)

df["nifti_path_resolved"] = resolved_paths

print("Missing NIfTI paths:", len(missing_rows))
if missing_rows:
    print("First missing rows:")
    display(df.loc[missing_rows[:10], ["subject_id", "nifti_path"]])

# Keep only rows with available MRI
df = df.dropna(subset=["nifti_path_resolved"]).copy()
df["nifti_path"] = df["nifti_path_resolved"]
df = df.drop(columns=["nifti_path_resolved"])

print("\nAfter dropping missing MRI rows")
print("Rows:", len(df))
print("Unique subjects:", df["subject_id"].nunique())
print("AD prevalence:", df["H"].mean())

In [ ]:
subject_summary = df.groupby("subject_id")["H"].agg(["count", "nunique", "mean"]).reset_index()

print("Subjects:", len(subject_summary))
print("Rows:", len(df))
print("Rows per subject:")
display(subject_summary["count"].describe())

mixed_subjects = subject_summary[subject_summary["nunique"] > 1]
print("Subjects with mixed labels:", len(mixed_subjects))

if len(mixed_subjects) > 0:
    print("Warning: some subjects have both H=0 and H=1 across rows.")
    display(mixed_subjects.head())

# Check class counts by subject-level label majority
print("\nRow-level H counts:")
print(df["H"].value_counts())

print("\nSubject-level approximate H counts:")
print(df.groupby("subject_id")["H"].first().value_counts())

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img = nib.load(row["nifti_path"]).get_fdata().astype("float32")
        img = torch.from_numpy(img).unsqueeze(0)  # [1, D, H, W]

        # Resize to 64^3
        img = F.interpolate(
            img.unsqueeze(0),
            size=(64, 64, 64),
            mode="trilinear",
            align_corners=False
        ).squeeze(0)

        # Per-volume z-score normalization
        img = (img - img.mean()) / (img.std() + 1e-5)

        label = torch.tensor(float(row["H"]), dtype=torch.float32)

        return img, label

In [ ]:
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv3d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv3d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool3d(2)

        self.fc1 = nn.Linear(16 * 16 * 16 * 16, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [B, 8, 32, 32, 32]
        x = self.pool(F.relu(self.conv2(x)))  # [B, 16, 16, 16, 16]
        x = x.reshape(x.size(0), -1)

        x = F.relu(self.fc1(x))
        logits = self.fc2(x).squeeze(1)

        return logits

In [ ]:
def train_one_fold(train_df, seed):
    set_seed(seed)

    train_ds = MRIDataset(train_df)
    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    model = Simple3DCNN().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for X, y in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            X = X.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        mean_loss = total_loss / max(len(train_dl), 1)
        print(f"Epoch {epoch}/{EPOCHS} | loss={mean_loss:.4f}")

    return model


def predict_dataframe(model, valid_df):
    valid_ds = MRIDataset(valid_df)
    valid_dl = DataLoader(
        valid_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    model.eval()
    preds = []

    with torch.no_grad():
        for X, _ in tqdm(valid_dl, desc="Predicting", leave=False):
            X = X.to(DEVICE)
            logits = model(X)
            prob = torch.sigmoid(logits).cpu().numpy()
            preds.extend(prob.tolist())

    return np.asarray(preds, dtype=float)

In [ ]:
def make_subject_level_oof_predictions(df):
    df = df.copy().reset_index(drop=True)

    oof_pred = np.full(len(df), np.nan, dtype=float)
    oof_fold = np.full(len(df), -1, dtype=int)

    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED
    )

    for fold, (train_idx, valid_idx) in enumerate(
        splitter.split(
            df,
            y=df["H"].astype(int),
            groups=df["subject_id"].astype(str)
        ),
        start=1
    ):
        print("\n" + "=" * 80)
        print(f"Fold {fold}/{N_FOLDS}")
        print("=" * 80)

        train_df = df.iloc[train_idx].reset_index(drop=True)
        valid_df = df.iloc[valid_idx].reset_index(drop=True)

        train_subjects = set(train_df["subject_id"].astype(str))
        valid_subjects = set(valid_df["subject_id"].astype(str))
        overlap = train_subjects & valid_subjects

        print("Train rows:", len(train_df))
        print("Valid rows:", len(valid_df))
        print("Train subjects:", len(train_subjects))
        print("Valid subjects:", len(valid_subjects))
        print("Subject overlap:", len(overlap))
        print("Train AD prevalence:", train_df["H"].mean())
        print("Valid AD prevalence:", valid_df["H"].mean())

        if len(overlap) > 0:
            raise RuntimeError(
                f"Subject leakage detected in fold {fold}: "
                f"{len(overlap)} overlapping subjects."
            )

        model = train_one_fold(train_df, seed=SEED + fold)
        preds = predict_dataframe(model, valid_df)

        if len(preds) != len(valid_idx):
            raise RuntimeError(
                f"Prediction length mismatch in fold {fold}: "
                f"{len(preds)} predictions for {len(valid_idx)} validation rows."
            )

        oof_pred[valid_idx] = preds
        oof_fold[valid_idx] = fold

        y_valid = df.iloc[valid_idx]["H"].to_numpy(dtype=int)

        try:
            fold_auc = roc_auc_score(y_valid, preds)
        except ValueError:
            fold_auc = np.nan

        yhat = (preds >= 0.5).astype(int)
        fold_acc = accuracy_score(y_valid, yhat)

        print(f"Fold AUC: {fold_auc:.4f}")
        print(f"Fold ACC @ 0.5: {fold_acc:.4f}")

    if np.isnan(oof_pred).any():
        n_missing = int(np.isnan(oof_pred).sum())
        raise RuntimeError(f"{n_missing} rows did not receive OOF predictions.")

    out = df.copy()
    out["autorater_prediction"] = oof_pred
    out["oof_fold"] = oof_fold
    out["A_prob"] = out["autorater_prediction"].astype(float)
    out["A_class_0p5"] = (out["A_prob"] >= 0.5).astype(int)

    return out

In [ ]:
out_df = make_subject_level_oof_predictions(df)

In [ ]:
y = out_df["H"].to_numpy(dtype=int)
p = out_df["autorater_prediction"].to_numpy(dtype=float)
yhat = (p >= 0.5).astype(int)

overall_auc = roc_auc_score(y, p)
overall_acc = accuracy_score(y, yhat)

tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
tpr = tp / (tp + fn) if (tp + fn) else np.nan
tnr = tn / (tn + fp) if (tn + fp) else np.nan

print("=== Overall subject-level OOF metrics ===")
print(f"AUC: {overall_auc:.4f}")
print(f"ACC @ 0.5: {overall_acc:.4f}")
print(f"TPR @ 0.5: {tpr:.4f}")
print(f"TNR @ 0.5: {tnr:.4f}")

print("\nOOF fold counts:")
print(out_df["oof_fold"].value_counts().sort_index())

print("\nSubject overlap check by fold:")
for fold in sorted(out_df["oof_fold"].unique()):
    fold_subjects = set(out_df.loc[out_df["oof_fold"] == fold, "subject_id"].astype(str))
    other_subjects = set(out_df.loc[out_df["oof_fold"] != fold, "subject_id"].astype(str))
    overlap = fold_subjects & other_subjects
    print(f"Fold {fold}: overlap with other folds = {len(overlap)}")

In [ ]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
out_df.to_csv(OUT_CSV, index=False)

print("Saved:")
print(OUT_CSV)

print("\nFinal columns:")
print(out_df.columns.tolist())

display(out_df.head())

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# =========================
# Paths
# =========================

OOF_CSV = REPO_ROOT / "results/autorater/autorater_predictions_all_subject_oof.csv"
META_CSV = REPO_ROOT / "data/csv/all_people_7_02_2026.csv"

OUT_MERGED = REPO_ROOT / "results/autorater/autorater_predictions_all_subject_oof_with_scan_age.csv"
OUT_6570 = REPO_ROOT / "results/autorater/autorater_predictions_6570_subject_oof_scan_age.csv"

# =========================
# Load files
# =========================

oof = pd.read_csv(OOF_CSV).copy()
meta = pd.read_csv(META_CSV).copy()

oof.columns = [c.strip() for c in oof.columns]
meta.columns = [c.strip() for c in meta.columns]

print("OOF shape:", oof.shape)
print("OOF columns:", oof.columns.tolist())

print("\nMetadata shape:", meta.shape)
print("Metadata columns:", meta.columns.tolist())

# =========================
# Required columns
# =========================

if "subject_id" not in oof.columns:
    raise ValueError("OOF file must contain subject_id.")

if "Subject" not in meta.columns:
    raise ValueError("Metadata file must contain Subject.")

if "Acq_Date" not in oof.columns:
    raise ValueError("OOF file must contain Acq_Date.")

if "Acq Date" not in meta.columns:
    raise ValueError("Metadata file must contain Acq Date.")

age_candidates = ["Age", "AGE", "age", "Age_at_scan", "age_at_scan", "AGE_AT_SCAN"]
age_col = next((c for c in age_candidates if c in meta.columns), None)

if age_col is None:
    raise ValueError(f"Could not find Age column. Tried: {age_candidates}")

print("\nUsing metadata subject column: Subject")
print("Using OOF date column: Acq_Date")
print("Using metadata date column: Acq Date")
print("Using metadata age column:", age_col)

# =========================
# Normalize subject IDs
# =========================

def normalize_subject_id(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().upper()

    # ADNI subject-ID pattern: NNN_S_NNNN
    m = re.search(r"(\d{3}_S_\d{4})", s)
    if m:
        return m.group(1)

    s = re.sub(r"^SUB[-_]", "", s)
    return s

oof["subject_norm"] = oof["subject_id"].apply(normalize_subject_id)
meta["subject_norm"] = meta["Subject"].apply(normalize_subject_id)

# =========================
# Normalize dates
# =========================

oof["acq_date_norm"] = pd.to_datetime(oof["Acq_Date"], errors="coerce").dt.date
meta["acq_date_norm"] = pd.to_datetime(meta["Acq Date"], errors="coerce").dt.date

meta[age_col] = pd.to_numeric(meta[age_col], errors="coerce")

print("\nOOF missing normalized dates:", oof["acq_date_norm"].isna().sum())
print("Metadata missing normalized dates:", meta["acq_date_norm"].isna().sum())
print("Metadata missing age:", meta[age_col].isna().sum())

# =========================
# Prepare scan-level metadata age table
# =========================

meta_scan_age = meta.dropna(
    subset=["subject_norm", "acq_date_norm", age_col]
).copy()

# There may be duplicate rows for the same subject/date.
# Aggregate to one row per subject/date.
scan_age_map = (
    meta_scan_age
    .groupby(["subject_norm", "acq_date_norm"], as_index=False)
    .agg(
        Age=(age_col, "median"),
        age_min=(age_col, "min"),
        age_max=(age_col, "max"),
        age_n=(age_col, "count"),
        meta_group=("Group", lambda x: ",".join(sorted(set(map(str, x)))) if "Group" in meta_scan_age.columns else "")
    )
)

print("\nUnique OOF subject-date pairs:", oof[["subject_norm", "acq_date_norm"]].drop_duplicates().shape[0])
print("Unique metadata subject-date pairs:", scan_age_map[["subject_norm", "acq_date_norm"]].drop_duplicates().shape[0])

dup_subject_date = int((scan_age_map["age_n"] > 1).sum())
print("Metadata subject-date pairs with duplicate entries:", dup_subject_date)

if dup_subject_date > 0:
    print("\nExamples of duplicate subject-date metadata rows:")
    display(scan_age_map[scan_age_map["age_n"] > 1].head(10))

# =========================
# Exact merge by subject + acquisition date
# =========================

merged = oof.merge(
    scan_age_map,
    on=["subject_norm", "acq_date_norm"],
    how="left"
)

missing_age = int(merged["Age"].isna().sum())

print("\nAfter exact subject+date merge")
print("Rows:", len(merged))
print("Rows missing Age:", missing_age)
print("Percent missing Age:", f"{100 * missing_age / len(merged):.2f}%")

if missing_age > 0:
    print("\nExamples of rows missing exact scan-level Age:")
    display(
        merged.loc[merged["Age"].isna(), ["subject_id", "Acq_Date", "subject_norm", "acq_date_norm"]]
        .drop_duplicates()
        .head(20)
    )

# =========================
# Optional fallback for missing exact dates
# =========================
# If there are a few unmatched rows due to date formatting or metadata mismatch,
# use nearest acquisition date within the same subject.
# This keeps a traceable fallback instead of silently using subject median.

def nearest_age_for_row(row, meta_scan_age):
    if pd.notna(row["Age"]):
        return row["Age"], row["age_min"], row["age_max"], row["age_n"], 0

    sid = row["subject_norm"]
    d = row["acq_date_norm"]

    if pd.isna(sid) or pd.isna(d):
        return np.nan, np.nan, np.nan, np.nan, np.nan

    cand = meta_scan_age[meta_scan_age["subject_norm"] == sid].copy()
    if len(cand) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    cand["date_diff_days"] = cand["acq_date_norm"].apply(lambda x: abs((x - d).days))
    best = cand.sort_values("date_diff_days").iloc[0]

    return (
        float(best[age_col]),
        float(best[age_col]),
        float(best[age_col]),
        1,
        int(best["date_diff_days"])
    )

if missing_age > 0:
    print("\nApplying nearest-date fallback for missing rows.")

    fallback_values = merged.apply(
        lambda row: nearest_age_for_row(row, meta_scan_age),
        axis=1,
        result_type="expand"
    )
    fallback_values.columns = [
        "Age_fallback",
        "age_min_fallback",
        "age_max_fallback",
        "age_n_fallback",
        "age_date_diff_days"
    ]

    merged = pd.concat([merged, fallback_values], axis=1)

    for col in ["Age", "age_min", "age_max", "age_n"]:
        fcol = f"{col}_fallback"
        merged[col] = merged[col].where(merged[col].notna(), merged[fcol])

    merged["age_match_type"] = np.where(
        merged["age_date_diff_days"].isna(),
        "exact",
        np.where(merged["age_date_diff_days"] == 0, "exact", "nearest_date")
    )

else:
    merged["age_match_type"] = "exact"
    merged["age_date_diff_days"] = 0

missing_after = int(merged["Age"].isna().sum())
print("\nAfter fallback")
print("Rows missing Age:", missing_after)
print("Age match types:")
print(merged["age_match_type"].value_counts(dropna=False))

if "age_date_diff_days" in merged.columns:
    print("\nNearest-date fallback day differences:")
    print(merged["age_date_diff_days"].describe())

# =========================
# Save full OOF table with scan-level Age
# =========================

OUT_MERGED.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUT_MERGED, index=False)

print("\nSaved merged OOF with scan-level Age:")
print(OUT_MERGED)

# =========================
# Create 65–70 scan-level subset
# =========================

subset_6570 = merged[
    (merged["Age"] >= 65)
    & (merged["Age"] <= 70)
].copy()

subset_6570.to_csv(OUT_6570, index=False)

print("\nSaved 65–70 scan-level subject-level OOF subset:")
print(OUT_6570)

print("\n65–70 subset summary")
print("Rows:", len(subset_6570))
print("Unique subjects:", subset_6570["subject_id"].nunique())
print("Age range:", subset_6570["Age"].min(), "to", subset_6570["Age"].max())

if "H" in subset_6570.columns:
    print("AD prevalence:", subset_6570["H"].mean())
    print("H counts:")
    print(subset_6570["H"].value_counts())

if "oof_fold" in subset_6570.columns:
    per_subject_fold_n = subset_6570.groupby("subject_id")["oof_fold"].nunique()
    print("Subjects spanning multiple OOF folds:", int((per_subject_fold_n > 1).sum()))
    print("Max folds per subject:", int(per_subject_fold_n.max()))

display(subset_6570.head())